# Do Not Penalize the Intercept

**Topic:** Ridge regression &nbsp;·&nbsp; **Difficulty:** Medium

Ridge regression in closed form is one line. Getting it *right* takes two,
because the intercept must not be shrunk: penalizing it means the fitted model is
no longer invariant to shifting the target, so adding 1000 to every `y` changes
your predictions by something other than 1000.

Fit the whole regularization path, pick lambda on a validation split, and confirm
the shift-invariance property holds.

## The data (created in the next cell)

- `X_train` `(200, 50)`, `y_train`: the target has a large offset (~500).
- `X_val` `(100, 50)`, `y_val` for choosing lambda.
- `X_test` `(200, 50)`, `y_test` for the final score.
- Features are correlated, so plain least squares is unstable here.

## Your task

With **NumPy only** (no sklearn):

1. For each lambda in `lambdas`, fit ridge **with an unpenalized intercept**.
   The clean way: centre `X` and `y` using training means, solve
   `(X^T X + lambda I) w = X^T y` on the centred data, then recover
   `b = mean_y - mean_X @ w`.
2. Pick the lambda with the lowest validation RMSE.
3. Set `answers` to a dict with `"val_rmse"` (one RMSE per lambda, same order),
   `"best_lambda"`, `"test_rmse"` (at the best lambda), and `"shift_ok"`: `1.0`
   if refitting on `y_train + 1000` moves every test prediction by 1000 to within
   `1e-6`, else `0.0`.

## Evaluation

Compared to the reference to a relative tolerance of 1e-6. `shift_ok` must be
`1.0`.

In [ ]:
import numpy as np

rng = np.random.default_rng(303)
d = 50
A = rng.normal(0, 1, (d, d))
cov = A @ A.T / d + 0.05 * np.eye(d)        # correlated features
L = np.linalg.cholesky(cov)

def make(n, seed):
    r = np.random.default_rng(seed)
    Z = r.normal(0, 1, (n, d)) @ L.T
    w = np.linspace(2.0, -2.0, d)
    return Z, Z @ w + 500.0 + r.normal(0, 3.0, n)

X_train, y_train = make(200, 11)
X_val, y_val = make(100, 12)
X_test, y_test = make(200, 13)
lambdas = np.logspace(-3, 4, 25)
print(f"train {X_train.shape}, mean y = {y_train.mean():.1f}, "
      f"condition number = {np.linalg.cond(X_train.T @ X_train):.1e}")

In [ ]:
# =====================================================================
# YOUR SOLUTION GOES HERE
# =====================================================================
# Use the training data created above to build your model, then create
# the variable(s) the problem statement asks for (see "Your task").
# When you are done, run the evaluation cell below to check your work.

In [ ]:
# =====================================================================
# EVALUATION: run this cell to score your answer
# =====================================================================
import numpy as np

if "answers" not in dir():
    raise NameError("Define `answers` in the solution cell before running this cell.")
if not isinstance(answers, dict):
    raise TypeError("`answers` must be a dict.")

def _digest(a):
    a = np.asarray(a, dtype=float).ravel()
    if a.size == 0:
        return 0.0
    probe = np.random.default_rng(9176).standard_normal(a.size)
    return float(a @ probe)

_CHECKS = [
    ("val_rmse", [25], 110.00921406138804, 18.94785671604582, 1e-06),
    ("best_lambda", [], 12.115276586285901, -2.0285578605362935, 1e-06),
    ("test_rmse", [], 3.3758631805623542, -0.5652478292387217, 1e-06),
    ("shift_ok", [], 1.0, -0.16743801481449916, 1e-06),
]

print("=" * 56)
_ok = True
for _key, _shape, _sum, _probe, _tol in _CHECKS:
    if _key not in answers:
        print(f"  {_key:<24} MISSING")
        _ok = False
        continue
    _a = np.asarray(answers[_key], dtype=float)
    if list(_a.shape) != _shape:
        print(f"  {_key:<24} shape {list(_a.shape)}, expected {_shape}")
        _ok = False
        continue
    if not np.all(np.isfinite(_a)):
        print(f"  {_key:<24} contains NaN or inf")
        _ok = False
        continue
    _g = (abs(_a.sum() - _sum) <= _tol * (1.0 + abs(_sum))
          and abs(_digest(_a) - _probe) <= _tol * (1.0 + abs(_probe)))
    print(f"  {_key:<24} {'ok' if _g else 'wrong'}")
    _ok = _ok and _g

if "shift_ok" in answers:
    _s = float(answers["shift_ok"]) == 1.0
    print(f"  {'intercept unpenalized':<24} {'ok' if _s else 'wrong'}")
    _ok = _ok and _s

print("=" * 56)
if _ok:
    print("PASSED: nice work!")
else:
    print("NOT PASSED yet: revisit your solution and try again.")

## Solution

Centring is what separates the intercept from the slopes: on centred data the
penalty applies only to `w`, and the intercept falls out afterwards as
`mean_y - mean_X @ w`. Because that expression is linear in `mean_y`, adding a
constant to `y` shifts only the intercept, which is precisely the invariance the
naive `(X^T X + lambda I)` on an augmented column of ones destroys.

```python
import numpy as np

def fit_ridge(X, y, lam):
    mx, my = X.mean(axis=0), y.mean()
    Xc, yc = X - mx, y - my
    w = np.linalg.solve(Xc.T @ Xc + lam * np.eye(X.shape[1]), Xc.T @ yc)
    return w, float(my - mx @ w)

def rmse(p, t):
    return float(np.sqrt(np.mean((p - t) ** 2)))

val_rmse = []
for lam in lambdas:
    w, b = fit_ridge(X_train, y_train, lam)
    val_rmse.append(rmse(X_val @ w + b, y_val))
val_rmse = np.array(val_rmse)

best_lambda = float(lambdas[int(np.argmin(val_rmse))])
w, b = fit_ridge(X_train, y_train, best_lambda)
test_rmse = rmse(X_test @ w + b, y_test)

w2, b2 = fit_ridge(X_train, y_train + 1000.0, best_lambda)
shift = (X_test @ w2 + b2) - (X_test @ w + b)
shift_ok = 1.0 if np.allclose(shift, 1000.0, atol=1e-6) else 0.0

answers = {"val_rmse": val_rmse, "best_lambda": best_lambda,
           "test_rmse": test_rmse, "shift_ok": shift_ok}
```